In [1]:
import kagglehub
import shutil
import os

# Download dataset (goes to kagglehub cache)
path = kagglehub.dataset_download("salader/dogsvscats")

# Target directory in /content
target_path = "/content/dogs_vs_cats"

# Remove target if it already exists (important)
if os.path.exists(target_path):
    shutil.rmtree(target_path)

# Copy dataset to /content
shutil.copytree(path, target_path)

print("Dataset copied to:", target_path)
print("Files:", os.listdir(target_path))

Using Colab cache for faster access to the 'dogsvscats' dataset.
Dataset copied to: /content/dogs_vs_cats
Files: ['catsvsdogs', 'train', 'test']


In [3]:
import tensorflow
from tensorflow import keras
from keras import Sequential ,layers
from keras.layers import Dense , Flatten
from keras.applications.vgg16 import VGG16

In [4]:
conv_base = VGG16(
    weights = "imagenet",
    include_top = False,
    input_shape = (150,150,3)
)

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [5]:
conv_base.trainable = True

In [9]:
conv_base.layers

[<InputLayer name=input_layer, built=True>,
 <Conv2D name=block1_conv1, built=True>,
 <Conv2D name=block1_conv2, built=True>,
 <MaxPooling2D name=block1_pool, built=True>,
 <Conv2D name=block2_conv1, built=True>,
 <Conv2D name=block2_conv2, built=True>,
 <MaxPooling2D name=block2_pool, built=True>,
 <Conv2D name=block3_conv1, built=True>,
 <Conv2D name=block3_conv2, built=True>,
 <Conv2D name=block3_conv3, built=True>,
 <MaxPooling2D name=block3_pool, built=True>,
 <Conv2D name=block4_conv1, built=True>,
 <Conv2D name=block4_conv2, built=True>,
 <Conv2D name=block4_conv3, built=True>,
 <MaxPooling2D name=block4_pool, built=True>,
 <Conv2D name=block5_conv1, built=True>,
 <Conv2D name=block5_conv2, built=True>,
 <Conv2D name=block5_conv3, built=True>,
 <MaxPooling2D name=block5_pool, built=True>]

In [10]:
#We are unfreezing last layers because they are specific for data and we will train them on our own dataset
set_trainable = False

for layer in conv_base.layers:
  if layer.name == "block5_conv1":
    set_trainable = True
  if set_trainable:
    layer.trainable = True
  else:
    layer.trainable = False
for layer in conv_base.layers:
  print(layer.name,layer.trainable)

input_layer False
block1_conv1 False
block1_conv2 False
block1_pool False
block2_conv1 False
block2_conv2 False
block2_pool False
block3_conv1 False
block3_conv2 False
block3_conv3 False
block3_pool False
block4_conv1 False
block4_conv2 False
block4_conv3 False
block4_pool False
block5_conv1 True
block5_conv2 True
block5_conv3 True
block5_pool True


In [11]:
model = Sequential()

model.add(conv_base)
model.add(Flatten())
model.add(Dense(256,activation='relu'))
model.add(Dense(1,activation='sigmoid'))

In [12]:

train_ds = keras.utils.image_dataset_from_directory(
    directory = "/content/dogs_vs_cats/train",
    batch_size = 32,
    subset = "training",
    validation_split = 0.2,
    image_size = (150,150),
    seed = 42
)

val_ds = keras.utils.image_dataset_from_directory(
    directory = "/content/dogs_vs_cats/train",
    batch_size = 32,
    subset = "validation",
    validation_split = 0.2,
    image_size = (150,150),
    seed = 42
)
test_ds = keras.utils.image_dataset_from_directory(
    directory = "/content/dogs_vs_cats/test",
    batch_size = 32,
    image_size = (150,150)
)

Found 20000 files belonging to 2 classes.
Using 16000 files for training.
Found 20000 files belonging to 2 classes.
Using 4000 files for validation.
Found 5000 files belonging to 2 classes.


In [13]:
def process(image, label):
    image = tensorflow.cast(image/255., tensorflow.float32)
    return image, label

train_ds = train_ds.map(process).prefetch(tensorflow.data.AUTOTUNE)
val_ds   = val_ds.map(process).prefetch(tensorflow.data.AUTOTUNE)

In [15]:
model.compile(
    optimizer = keras.optimizers.RMSprop(1e-5),
    loss = "binary_crossentropy",
    metrics = ["accuracy"]
)


In [16]:
model.fit(train_ds, validation_data = val_ds ,epochs=10)

Epoch 1/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 84s 144ms/step - accuracy: 0.8373 - loss: 0.3570 - val_accuracy: 0.9327 - val_loss: 0.1743
Epoch 2/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 70s 141ms/step - accuracy: 0.9383 - loss: 0.1532 - val_accuracy: 0.9410 - val_loss: 0.1474
Epoch 3/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 70s 140ms/step - accuracy: 0.9557 - loss: 0.1104 - val_accuracy: 0.9433 - val_loss: 0.1433
Epoch 4/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 70s 140ms/step - accuracy: 0.9732 - loss: 0.0763 - val_accuracy: 0.9470 - val_loss: 0.1374
Epoch 5/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 70s 140ms/step - accuracy: 0.9827 - loss: 0.0513 - val_accuracy: 0.9383 - val_loss: 0.1823
Epoch 6/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 70s 140ms/step - accuracy: 0.9892 - loss: 0.0381 - val_accuracy: 0.9498 - val_loss: 0.1455
Epoch 7/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 70s 140ms/step - accuracy: 0.9945 - loss: 0.0234 - val_accuracy: 0.9475 - val_loss: 0.1595
Epoch 8/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 70s 141ms/step - accuracy: 0.9970 - loss: 0

In [17]:
model.evaluate(test_ds)

157/157 ━━━━━━━━━━━━━━━━━━━━ 18s 117ms/step - accuracy: 0.9043 - loss: 21.1567


[22.52008056640625, 0.8992000222206116]